# Token Usage Benchmark Analysis

Comparing token efficiency between two runs:
- **With Local Agent** — the main model calls a local MCP tool (coder server) to generate code
- **Without Local Agent** — the main model generates all code directly

Both runs used the same prompt: *Build a basic Python calculator*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

In [ ]:
CSV_PATH = Path('groq-logs-first-run-no-local-agent.csv')

KEY_LABELS = {
    'key_01kvn4r2mqfjj8ba4fzbg8mvxe': 'With Local Agent',
    'key_01kvqbx8y6fs2tx19jf3bmcsqd': 'Without Local Agent',
}
RUN_ORDER = ['With Local Agent', 'Without Local Agent']

# Error rows (non-200) carry an extra JSON column and are auto-skipped as bad lines.
df_raw = pd.read_csv(CSV_PATH, on_bad_lines='warn')
df_raw['created_at_dt'] = pd.to_datetime(df_raw['created_at'], unit='ms', utc=True)

# Filter to the most recent date in the dataset (today's runs) and successful requests only
most_recent_date = df_raw['created_at_dt'].dt.date.max()
df = df_raw[
    (df_raw['created_at_dt'].dt.date == most_recent_date) &
    (df_raw['status_code'] == 200)
].copy()

df['run'] = df['api_key_id'].map(KEY_LABELS)

print(f'Date: {most_recent_date}  |  Total successful rows: {len(df)}')
print(df.groupby('run', dropna=False).size())

In [ ]:
summary = (
    df.groupby('run')
    .agg(
        api_calls=('request_id', 'count'),
        total_input_tokens=('input_tokens', 'sum'),
        total_cached_tokens=('input_cached_tokens', 'sum'),
        total_output_tokens=('output_tokens', 'sum'),
        avg_output_per_call=('output_tokens', 'mean'),
        avg_time_to_completion=('time_to_completion', 'mean'),
    )
    .round(2)
    .reindex(RUN_ORDER, fill_value=0)
)
summary.index.name = 'Run'
summary['total_non_cached_input'] = (
    summary['total_input_tokens'] - summary['total_cached_tokens']
)
summary

In [ ]:
COLORS = ['#4C72B0', '#DD8452']

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle(
    'Token Usage Benchmark: Local Agent MCP vs Direct Inference',
    fontsize=14, fontweight='bold',
)


def labeled_bar(ax, values, title, ylabel):
    bars = ax.bar(RUN_ORDER, values, color=COLORS, width=0.5, edgecolor='white')
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, _: f'{int(x):,}')
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', length=0)
    for bar, val in zip(bars, values):
        label = f'{val:,.0f}' if val >= 1 else f'{val:.2f}'
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold',
        )


labeled_bar(axes[0, 0], summary['api_calls'].tolist(), 'Total API Calls', 'Calls')
labeled_bar(axes[0, 1], summary['total_output_tokens'].tolist(), 'Total Output Tokens', 'Tokens')

# Stacked bar: non-cached + cached input tokens
ax = axes[1, 0]
non_cached = summary['total_non_cached_input'].tolist()
cached = summary['total_cached_tokens'].tolist()
ax.bar(RUN_ORDER, non_cached, color=COLORS, width=0.5, label='Non-cached', edgecolor='white')
ax.bar(
    RUN_ORDER, cached, bottom=non_cached, color=COLORS,
    width=0.5, alpha=0.4, label='Cached', edgecolor='white', hatch='//',
)
ax.set_title('Total Input Tokens (Cached vs Non-cached)', fontweight='bold', pad=10)
ax.set_ylabel('Tokens')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', length=0)

labeled_bar(
    axes[1, 1],
    summary['avg_output_per_call'].tolist(),
    'Avg Output Tokens per Call',
    'Tokens / Call',
)

plt.tight_layout()
plt.savefig('token_usage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved -> token_usage_comparison.png')

## Interpretation Guide

| Metric | What it tells us |
|--------|------------------|
| **Total output tokens** | Primary signal — code generated locally doesn't appear in cloud model output |
| **Avg output per call** | Normalises for different call volumes; lower = more focused calls |
| **Total API calls** | Agentic loops make more calls — expected to be higher for the local agent |
| **Cached input tokens** | Repeated context in the agent loop gets cached, reducing effective cost |

### Hypothesis holds if:
- `total_output_tokens` is meaningfully **lower** for *With Local Agent*
- `avg_output_per_call` is **lower** for *With Local Agent*
- The extra API calls (agent loop overhead) don't negate the output token savings